In [1]:
import re
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch.nn as nn

In [2]:
df = pd.read_csv('/kaggle/input/datasets/bitext/bitext-gen-ai-chatbot-customer-support-dataset/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv')
df.head()
df.shape
df.columns
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [3]:
X = df['instruction']
y = df['intent']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)


In [4]:


def check_placeholders(text):
    # List of specific words or regex patterns to look for
    patterns = [
        r"YOUR_[A-Za-z_]+",  # Catches YOUR_DATASET, YOUR_FILE, etc.
        r"\bunknown\b",      # Catches the exact word "unknown"
        r"todo",             # Catches "todo"
        r"<.*?>",            # Catches anything in angle brackets like <insert path>
    ]
    
    found = []
    for pattern in patterns:
        # re.IGNORECASE makes it catch "UNKNOWN", "Unknown", etc.
        matches = re.findall(pattern, str(text), flags=re.IGNORECASE) 
        if matches:
            found.extend(matches)
            
    if found:
        print(f"⚠️ Placeholder alert! Found: {set(found)}")
    else:
        print("✅ All clear! No placeholders detected.")
check_placeholders(df)

✅ All clear! No placeholders detected.


In [5]:
df["intent"].value_counts()

intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
delivery_options             995
delete_account               995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64

In [6]:
df["instruction"].str.len().describe()
df

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...
26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [7]:
text = X_train.tolist()
test_text = X_test.tolist()
def tokenize(text):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

tokens = [tokenize(sentence) for sentence in text]
test_tokens = [tokenize(sentence) for sentence in test_text]

In [8]:
vocab = {
    "<PAD>":0,
    "<UNK>":1
}

for token_list in tokens:
    for word in token_list:
        if word not in vocab:
            vocab[word] = len(vocab)

def encode(token_list):
    return [vocab.get(word, vocab['<UNK>']) for word in token_list]

encoded_text = [encode(token_list) for token_list in tokens]
encoded_test_text = [encode(token_list) for token_list in test_tokens]
# print(f'{tokens[0]} : {encoded_text[0]}') 


In [9]:
max_length = max(len(x) for x in encoded_text)

padded_text = []
padded_test_text = []
for token_list in encoded_text:
    padded_length = max_length - len(token_list)
    token_list += [0] * (padded_length)
    padded_text.append(token_list)

for token_list in encoded_test_text:
    if len(token_list) >= max_length:
        token_list = token_list[:max_length]
    else:
        padded_length = max_length - len(token_list)
        token_list += [0] * (padded_length)
    padded_test_text.append(token_list)
    

# padded_text

In [10]:
labels = sorted(y_train.unique())
labels_to_id ={
    label:i
    for i, label in enumerate(labels)
}

id_to_labels ={
    i:label
    for label, i in labels_to_id.items()
}

encoded_labels = [labels_to_id[label] for label in y_train]
encoding_test_labels = [labels_to_id[label] for label in y_test]



In [11]:
X_train_tensor = torch.tensor(padded_text, dtype=torch.long)
X_test_tensor = torch.tensor(padded_test_text, dtype=torch.long)
y_train_tensor = torch.tensor(encoded_labels, dtype=torch.long)
y_test_tensor = torch.tensor(encoding_test_labels, dtype=torch.long)

In [12]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size =64)


In [13]:
class ChatbotNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, max_length, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.network = nn.Sequential(
            nn.Linear(max_length * embedding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.size(0), -1)
        return self.network(x)

In [14]:
vocab_size = len(vocab)
embedding_dim = 64
num_classes = len(labels_to_id)

model = ChatbotNN(
    vocab_size,
    embedding_dim,
    max_length,
    num_classes
)

print(model)

ChatbotNN(
  (embedding): Embedding(2558, 64, padding_idx=0)
  (network): Sequential(
    (0): Linear(in_features=1088, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=27, bias=True)
  )
)


In [15]:

X_batch, y_batch = next(iter(train_loader))
output = model(X_batch)


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), 0.01)
loss = criterion(output, y_batch)


In [17]:
epochs = 50
for epoch in range(epochs):
    total_loss = 0
    correct = 0
    total = 0
    for batch_x, batch_y in train_loader:
        output = model(batch_x)
        loss = criterion(output, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        predictions = output.argmax(dim=1)
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)
    accuracy = correct/total
    avg_loss = total_loss/ len(train_loader)
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {avg_loss:.4f} "
        f"Accuracy: {accuracy:.4f}"
    )

Epoch [1/50] Loss: 0.4831 Accuracy: 0.8575
Epoch [2/50] Loss: 0.0700 Accuracy: 0.9807
Epoch [3/50] Loss: 0.0475 Accuracy: 0.9890
Epoch [4/50] Loss: 0.0778 Accuracy: 0.9862
Epoch [5/50] Loss: 0.1291 Accuracy: 0.9829
Epoch [6/50] Loss: 0.0856 Accuracy: 0.9911
Epoch [7/50] Loss: 0.0596 Accuracy: 0.9936
Epoch [8/50] Loss: 0.0361 Accuracy: 0.9963
Epoch [9/50] Loss: 0.0459 Accuracy: 0.9959
Epoch [10/50] Loss: 0.0539 Accuracy: 0.9954
Epoch [11/50] Loss: 0.0758 Accuracy: 0.9954
Epoch [12/50] Loss: 0.1102 Accuracy: 0.9933
Epoch [13/50] Loss: 0.0471 Accuracy: 0.9970
Epoch [14/50] Loss: 0.0548 Accuracy: 0.9972
Epoch [15/50] Loss: 0.0421 Accuracy: 0.9977
Epoch [16/50] Loss: 0.0309 Accuracy: 0.9981
Epoch [17/50] Loss: 0.0843 Accuracy: 0.9985
Epoch [18/50] Loss: 0.0697 Accuracy: 0.9971
Epoch [19/50] Loss: 0.0602 Accuracy: 0.9975
Epoch [20/50] Loss: 0.0517 Accuracy: 0.9979
Epoch [21/50] Loss: 0.0326 Accuracy: 0.9984
Epoch [22/50] Loss: 0.0452 Accuracy: 0.9977
Epoch [23/50] Loss: 0.0462 Accuracy: 0.99

In [18]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)

        predictions = output.argmax(dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 98.88%


In [19]:


torch.save({
    "ar": model.state_dict(),
    "vocab": vocab,
    "labels_to_id": labels_to_id,
    "id_to_label": id_to_labels,
    "embedding_dim": 64,
    "num_classes": len(labels_to_id)
}, "/kaggle/working/chatbot_model.pth")

print("Model saved!")

Model saved!


In [20]:
print(output.shape)
print(output[0])
print(y_batch[0])

torch.Size([63, 27])
tensor([-663.9783, -523.7141, -620.6942, -487.1838, -462.4784, -218.3745,
        -181.7422, -316.9954,  -49.7820, -507.5366, -598.9715, -464.5434,
        -355.6713,  136.7973, -415.9395, -575.9103, -361.1415, -211.6852,
          -1.0253,  562.0674, -502.9120, -569.9766, -388.4429, -756.0375,
        -427.9566, -216.9178, -620.4865])
tensor(19)


In [21]:
from IPython.display import Audio, display

display(Audio(
    data=[0, 1] * 1000,
    rate=2000,
    autoplay=True
))

In [22]:
max_length

17

In [23]:
len(y_test_tensor)
len(X_test_tensor)

5375